# 🧠 YOLOv11m Fine-Tuning — Purplle Retail Detection

This notebook trains a **YOLOv11m** person detector fine-tuned on frames extracted from your Purplle store CCTV clips, then exports to **ONNX** for CPU inference.

## Prerequisites
1. **Runtime → Change runtime type → GPU (T4 or A100)**
2. Upload your CCTV video clips to Google Drive OR mount Drive and reference them directly
3. Run each cell in order — they are numbered and self-contained

## What gets produced
- `yolov11m_retail.onnx` — the model file to drop into `models/`
- `runs/detect/train/` — training metrics and weight checkpoints

In [ ]:
# ============================================================
# CELL 1 — Verify GPU is available
# ============================================================
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '⚠️  No GPU detected — switch runtime to GPU')

In [ ]:
# ============================================================
# CELL 2 — Install dependencies
# ============================================================
!pip install -q ultralytics==8.2.0 opencv-python-headless roboflow
import ultralytics
ultralytics.checks()
print(f'ultralytics version: {ultralytics.__version__}')

In [ ]:
# ============================================================
# CELL 3 — Mount Google Drive (your clips must be here)
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os

# ──────────────────────────────────────────────────────────────
# ⚙️  CONFIGURE: Set the path to where your clips are in Drive
# ──────────────────────────────────────────────────────────────
CLIPS_DIR = '/content/drive/MyDrive/purplle/Stores'   # <-- change this

assert os.path.exists(CLIPS_DIR), f'Clips directory not found: {CLIPS_DIR}'

# List all found video clips
clips = []
for root, dirs, files in os.walk(CLIPS_DIR):
    for f in files:
        if f.endswith(('.mp4', '.avi', '.mov')):
            clips.append(os.path.join(root, f))

print(f'Found {len(clips)} clips:')
for c in clips:
    print(' ', c)

In [ ]:
# ============================================================
# CELL 4 — Extract frames from all CCTV clips
# Samples 1 frame every N frames to get a representative dataset
# ============================================================
import cv2
import os

FRAME_SAMPLE_INTERVAL = 30   # Extract 1 frame every 30 frames (= 2fps at 60fps, 0.5fps at 15fps)
OUTPUT_IMAGES_DIR = '/content/dataset/images/all'
os.makedirs(OUTPUT_IMAGES_DIR, exist_ok=True)

frame_count = 0
for clip_path in clips:
    cap = cv2.VideoCapture(clip_path)
    clip_name = os.path.splitext(os.path.basename(clip_path))[0].replace(' ', '_')
    idx = 0
    saved = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if idx % FRAME_SAMPLE_INTERVAL == 0:
            fname = f'{clip_name}_f{idx:06d}.jpg'
            cv2.imwrite(os.path.join(OUTPUT_IMAGES_DIR, fname), frame)
            saved += 1
        idx += 1
    cap.release()
    frame_count += saved
    print(f'  {clip_name}: {saved} frames saved (total frames in clip: {idx})')

print(f'\nTotal frames extracted: {frame_count}')
print(f'Saved to: {OUTPUT_IMAGES_DIR}')

In [ ]:
# ============================================================
# CELL 5 — Auto-label using pretrained YOLOv11m (pseudo-labels)
#
# Strategy: Use the pretrained model to generate initial bounding
# box annotations. You can then review/fix labels in Label Studio
# or Roboflow, OR use pseudo-labels directly for fine-tuning
# (acceptable for retail detection where persons are abundant).
# ============================================================
from ultralytics import YOLO
import os, glob, shutil

LABELS_DIR = '/content/dataset/labels/all'
os.makedirs(LABELS_DIR, exist_ok=True)

# Load pretrained YOLOv11m
model = YOLO('yolo11m.pt')

image_files = sorted(glob.glob(os.path.join(OUTPUT_IMAGES_DIR, '*.jpg')))
print(f'Auto-labelling {len(image_files)} images...')

label_count = 0
for img_path in image_files:
    results = model(img_path, conf=0.3, classes=[0], verbose=False)  # class 0 = person
    img_name = os.path.splitext(os.path.basename(img_path))[0]
    label_path = os.path.join(LABELS_DIR, img_name + '.txt')
    
    with open(label_path, 'w') as f:
        for result in results:
            for box in result.boxes:
                cls = int(box.cls[0])  # should be 0 (person)
                xywhn = box.xywhn[0].tolist()  # normalised x_center, y_center, w, h
                f.write(f'{cls} {xywhn[0]:.6f} {xywhn[1]:.6f} {xywhn[2]:.6f} {xywhn[3]:.6f}\n')
    label_count += 1

print(f'Labels written: {label_count}')
print('TIP: Review labels in Label Studio or Roboflow before training for best accuracy.')

In [ ]:
# ============================================================
# CELL 6 — Split dataset 80/20 train/val
# ============================================================
import os, glob, shutil, random

random.seed(42)

for split in ['train', 'val']:
    os.makedirs(f'/content/dataset/images/{split}', exist_ok=True)
    os.makedirs(f'/content/dataset/labels/{split}', exist_ok=True)

all_images = sorted(glob.glob('/content/dataset/images/all/*.jpg'))
random.shuffle(all_images)

split_idx = int(len(all_images) * 0.8)
train_images = all_images[:split_idx]
val_images   = all_images[split_idx:]

def copy_split(image_list, split):
    for img_path in image_list:
        img_name = os.path.basename(img_path)
        lbl_name = os.path.splitext(img_name)[0] + '.txt'
        lbl_path = os.path.join('/content/dataset/labels/all', lbl_name)
        shutil.copy(img_path, f'/content/dataset/images/{split}/{img_name}')
        if os.path.exists(lbl_path):
            shutil.copy(lbl_path, f'/content/dataset/labels/{split}/{lbl_name}')

copy_split(train_images, 'train')
copy_split(val_images, 'val')

print(f'Train: {len(train_images)} images')
print(f'Val:   {len(val_images)} images')

In [ ]:
# ============================================================
# CELL 7 — Write dataset YAML config
# ============================================================
import yaml

dataset_yaml = {
    'path': '/content/dataset',
    'train': 'images/train',
    'val':   'images/val',
    'nc': 1,
    'names': ['person']
}

with open('/content/retail.yaml', 'w') as f:
    yaml.dump(dataset_yaml, f)

print('Dataset config:')
print(open('/content/retail.yaml').read())

In [ ]:
# ============================================================
# CELL 8 — Fine-tune YOLOv11m on retail frames
#
# Training params follow design.md §8.1:
#   epochs=50, imgsz=640, batch=8, conf=0.3
#
# Expected training time on T4:
#   ~500 frames  → ~10 min
#   ~2000 frames → ~40 min
#   ~5000 frames → ~90 min
# ============================================================
from ultralytics import YOLO

model = YOLO('yolo11m.pt')   # Start from pretrained weights

results = model.train(
    data='/content/retail.yaml',
    epochs=50,
    imgsz=640,
    batch=8,
    device=0,          # GPU
    conf=0.3,          # Low threshold — ByteTrack handles FP filtering
    iou=0.45,          # Soft-NMS equivalent
    cos_lr=True,       # Cosine annealing
    augment=True,      # Mosaic, color jitter, flip augmentation
    project='/content/runs/detect',
    name='yolov11m_retail',
    save=True,
    patience=15,       # Early stopping
    cache=True,        # Cache images in RAM for faster training
)

print('Training complete!')
print(f'Best mAP@0.5: {results.results_dict.get("metrics/mAP50(B)", "N/A")}')

In [ ]:
# ============================================================
# CELL 9 — Validate on held-out frames
# Target: mAP@0.5 > 0.70 for retail crowd detection
# ============================================================
from ultralytics import YOLO

best_model_path = '/content/runs/detect/yolov11m_retail/weights/best.pt'
model = YOLO(best_model_path)

metrics = model.val(data='/content/retail.yaml', conf=0.3, iou=0.45)

map50   = metrics.box.map50
map5095 = metrics.box.map

print(f'mAP@0.5:      {map50:.4f}  (target: > 0.70)')
print(f'mAP@0.5:0.95: {map5095:.4f}')

if map50 < 0.70:
    print('⚠️  mAP below target. Consider: more epochs, more diverse frames, or manual annotation review.')
else:
    print('✅  Model meets target accuracy. Proceeding to ONNX export.')

In [ ]:
# ============================================================
# CELL 10 — Export to ONNX
# Flags match design.md §8.1: dynamic=True, simplify=True
# ============================================================
from ultralytics import YOLO

best_model_path = '/content/runs/detect/yolov11m_retail/weights/best.pt'
model = YOLO(best_model_path)

onnx_path = model.export(
    format='onnx',
    dynamic=True,       # Variable batch size
    simplify=True,      # Graph optimisation via onnx-simplifier
    opset=17,
    imgsz=640,
)

print(f'ONNX model exported to: {onnx_path}')

# Verify ONNX model integrity
import onnx
m = onnx.load(onnx_path)
onnx.checker.check_model(m)
print('✅  ONNX model validation passed')

In [ ]:
# ============================================================
# CELL 11 — Benchmark ONNX inference speed on CPU
# Target: ≥ 10 fps per camera (design.md §Requirement 2.6)
# ============================================================
import onnxruntime as ort
import numpy as np
import time

onnx_path = '/content/runs/detect/yolov11m_retail/weights/best.onnx'
session = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])

# Warmup
dummy = np.random.rand(1, 3, 640, 640).astype(np.float32)
for _ in range(5):
    session.run(None, {session.get_inputs()[0].name: dummy})

# Benchmark 50 iterations
N = 50
t0 = time.perf_counter()
for _ in range(N):
    session.run(None, {session.get_inputs()[0].name: dummy})
elapsed = time.perf_counter() - t0

avg_ms = (elapsed / N) * 1000
fps = 1000 / avg_ms

print(f'Average inference time: {avg_ms:.1f} ms')
print(f'Throughput: {fps:.1f} fps')

if fps >= 10:
    print('✅  Meets ≥10 fps requirement on CPU')
else:
    print(f'⚠️  Below target. Consider quantisation (INT8) or reducing imgsz to 480.')

In [ ]:
# ============================================================
# CELL 12 — Copy ONNX model to Google Drive for download
# Then download it and place in your local models/ directory
# ============================================================
import shutil, os

onnx_src = '/content/runs/detect/yolov11m_retail/weights/best.onnx'
drive_dest = '/content/drive/MyDrive/purplle/models/yolov11m_retail.onnx'
os.makedirs(os.path.dirname(drive_dest), exist_ok=True)
shutil.copy(onnx_src, drive_dest)

print(f'✅  Model saved to Google Drive: {drive_dest}')
print()
print('Next steps:')
print('  1. Download yolov11m_retail.onnx from Google Drive')
print('  2. Place it in your local: models/yolov11m_retail.onnx')
print('  3. Restart the vision-pipeline container:')
print('     docker compose restart vision-pipeline')
print('  The pipeline will auto-detect and load the ONNX model on startup.')

# Also download directly from Colab if preferred
from google.colab import files
files.download(onnx_src)

## 📊 Training Summary

After running all cells:

| Item | Value |
|------|-------|
| Base model | YOLOv11m (pretrained COCO) |
| Dataset | Frames extracted from your 8 CCTV clips |
| Export format | ONNX (dynamic batch, simplified graph) |
| Target mAP@0.5 | > 0.70 |
| Target CPU speed | ≥ 10 fps |
| Output file | `yolov11m_retail.onnx` → place in `models/` |

### If mAP < 0.70:
- Increase epochs to 100
- Manually review/fix pseudo-labels for 50–100 difficult frames (dense crowds, heavy occlusion)
- Add frames from different lighting conditions (morning vs evening lighting)
- Reduce `FRAME_SAMPLE_INTERVAL` in Cell 4 to extract more frames

### If speed < 10 fps:
```python
# INT8 quantisation (run after Cell 10):
!pip install -q onnxconverter-common
from onnxconverter_common import float16
import onnx
model_fp32 = onnx.load('best.onnx')
model_fp16 = float16.convert_float_to_float16(model_fp32)
onnx.save(model_fp16, 'best_fp16.onnx')
```